# Standard Scores

*Express distance from the mean in standard-deviation units and interpret threshold flags carefully.*


## Center and Rescale

With known population parameters, a standard score is $z_i=(x_i-\mu)/\sigma$.
For the observed sample here, use $(x_i-\bar{x})/s$ with `ddof=1` consistently.
The result is dimensionless: a positive value is above the mean and a negative
value is below it. Subtracting and dividing does not make a distribution normal.


In [1]:
import numpy as np
import pandas as pd

values = np.array([1, 3, 5, 7, 9], dtype=float)
sample_std = values.std(ddof=1)
z_scores = (values - values.mean()) / sample_std
print(
    pd.DataFrame({"value": values, "z_score": z_scores}).round(4).to_string(index=False)
)
print("Mean near zero:", np.isclose(z_scores.mean(), 0))
print("Sample standard deviation near one:", np.isclose(z_scores.std(ddof=1), 1))


 value  z_score
   1.0  -1.2649
   3.0  -0.6325
   5.0   0.0000
   7.0   0.6325
   9.0   1.2649
Mean near zero: True
Sample standard deviation near one: True


The value 9 is about 1.2649 sample standard deviations above the mean.
The transformed mean is zero and its standard deviation is one under the same
`ddof=1` convention.


## The 68–95–99.7 Rule Has a Distribution Assumption

For a normal population, approximately 68%, 95%, and 99.7% of its probability
lies within one, two, and three population standard deviations of its mean.
`norm.cdf()` evaluates the cumulative probability of a standard normal variable.
These percentages are not guarantees for every sample or every distribution.
Neither symmetry nor a sample size above 30 establishes normality.


In [2]:
from scipy.stats import norm

for distance in [1, 2, 3]:
    probability = norm.cdf(distance) - norm.cdf(-distance)
    print(
        f"Normal probability within {distance} standard deviations: {100 * probability:.2f}%"
    )


Normal probability within 1 standard deviations: 68.27%
Normal probability within 2 standard deviations: 95.45%
Normal probability within 3 standard deviations: 99.73%


The computed probabilities are approximately 68.27%, 95.45%, and 99.73%.
They describe the normal model, not an empirical count in the five observations.


## A Threshold Produces Candidates for Review

A common inspection rule flags $|z|>3$. Large observations also change the mean
and standard deviation used to compute their own scores, so the rule can miss
extremes, especially in small samples. Keep the raw observations and inspect
flagged values before choosing a treatment.


In [3]:
measurements = np.array([10, 11, 12] * 10 + [100], dtype=float)
measurement_z = (measurements - measurements.mean()) / measurements.std(ddof=1)
flagged = np.abs(measurement_z) > 3
print("Flagged values:", measurements[flagged])
print("Flagged standard scores:", np.round(measurement_z[flagged], 3))
print("Total observations retained:", measurements.size)


Flagged values: [100.]
Flagged standard scores: [5.381]
Total observations retained: 31


The value 100 is flagged, and all 31 observations remain available.
The flag identifies a large standardized distance; it does not establish why
the value was recorded.


## Constant Data Cannot Be Standardized This Way

If every value is equal, the standard deviation is zero. Division by that value
is undefined. Check it before calculating standard scores.


In [4]:
constant_values = np.array([7, 7, 7], dtype=float)
constant_std = constant_values.std(ddof=1)
if constant_std == 0:
    print("Standard scores are undefined: every value is identical")
else:
    print((constant_values - constant_values.mean()) / constant_std)


Standard scores are undefined: every value is identical


The constant sample receives an explicit explanation instead of a divide-by-zero warning or invented standard scores.


A standard score measures relative distance, not error or probability by itself. State the standard-deviation convention and inspect distribution assumptions before using a threshold.

[Related explanation in the source lecture](https://www.youtube.com/watch?v=9bM65kxOwPA&t=2404s).
